# Inspect Logged Runs And Artifacts

Review prior AutoML runs, compare candidates, load a winning model, and verify the logged data/model artifacts from MLflow and GCS.

In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from mlflow.tracking import MlflowClient

import automl
from automl import data, eval, experiment, trial
from automl.mlflow import client as automl_mlflow_client
from automl.mlflow import trial as mlflow_trial
from automl.utils.io import gcs

DRY_RUN = True
NAMESPACE = os.getenv("AUTOML_NOTEBOOK_NAMESPACE", "")


In [ ]:
active = automl.use_project(dry_run=DRY_RUN, namespace=NAMESPACE)
config = active.config
display(
    {
        "project": active.project_name,
        "repo_root": str(config.repo_root),
        "project_dir": str(config.project_dir),
        "experiment": active.active_experiment_id,
        "dry_run": active.dry_run,
        "namespace": active.namespace or "<none>",
    }
)



def message_frame(message: str) -> pd.DataFrame:
    return pd.DataFrame({"message": [message]})


{
    "python": sys.executable,
    "project": active.project_name,
    "repo_root": str(config.repo_root),
    "project_dir": str(config.project_dir),
    "target": config.target_column,
}


## Experiment Overview

The setup cell asks AutoML to resolve the active project from the kernel `cwd`. In VS Code this repo sets notebooks to start from their own folder, so AutoML can crawl up to `projects/example_homecredit/config.py` without a project variable in the notebook.

In [ ]:
experiments_df = pd.DataFrame(
    [
        {
            "project": active.project_name,
            "experiment": active.active_experiment_id,
            "dry_run": active.dry_run,
            "namespace": active.namespace or "<none>",
            "mlflow_tracking_uri": config.mlflow_tracking_uri,
        }
    ]
)
experiments_df


In [ ]:
leaderboard = experiment.leaderboard(n=20, session=active)
leaderboard_rows = [
    {
        **row.to_dict(),
        "strategy": row.strategy,
        "hypothesis": row.hypothesis,
        "validation": row.status.value,
    }
    for row in leaderboard.rows
]
leaderboard_columns = [
    "slug",
    "primary_metric_value",
    "strategy",
    "training_origin",
    "validation",
    "run_id",
]
leaderboard_df = pd.DataFrame(leaderboard_rows).reindex(columns=leaderboard_columns)

leaderboard_df.head(10) if not leaderboard_df.empty else message_frame(
    "No successful dry-run trials yet. Start with 3.1_run_agent_automl.ipynb or 3.2_author_new_trial.ipynb."
)


## Compare Real Alternatives

`experiment.compare` compares MLflow run IDs. To avoid comparing duplicate reruns of the same trial family, this cell selects the best two distinct slugs from the leaderboard.

In [ ]:
comparison_candidates = []
seen_slugs = set()
for row in leaderboard.rows:
    if row.slug in seen_slugs:
        continue
    comparison_candidates.append(row)
    seen_slugs.add(row.slug)
    if len(comparison_candidates) == 2:
        break

candidate_df = pd.DataFrame(
    [
        {
            "slot": index,
            "slug": row.slug,
            "run_id": row.run_id,
            "primary_metric": row.primary_metric_value,
            "strategy": row.strategy,
            "n_features": row.n_features,
            "fit_seconds": row.training_time_s,
        }
        for index, row in enumerate(comparison_candidates, start=1)
    ]
).reindex(columns=["slot", "slug", "run_id", "primary_metric", "strategy", "n_features", "fit_seconds"])

display(candidate_df if not candidate_df.empty else message_frame("No completed trials to compare yet."))

if len(comparison_candidates) >= 2:
    comparison = experiment.compare([row.run_id for row in comparison_candidates], session=active)
    comparison_df = pd.DataFrame([delta.to_dict() for delta in comparison.metric_deltas])
    comparison_view = comparison_df[
        comparison_df["metric"].isin([
            "auc",
            "n_features",
            "time.fit_seconds",
            "time.predict_seconds",
            "validation.latency_p50_ms",
            "validation.max_abs_diff",
        ])
    ]
else:
    comparison = None
    comparison_df = pd.DataFrame(columns=["metric", "value_a", "value_b", "delta"])
    comparison_view = message_frame(
        "Need at least two distinct successful trials before metric comparison is available."
    )

comparison_view


## Load The Winner

Load the winning MLflow PyFunc model, unwrap the underlying Python model, and inspect the model-owned training state.

In [ ]:
winner = comparison_candidates[0] if comparison_candidates else (leaderboard.rows[0] if leaderboard.rows else None)
winner_summary = None
loaded_model = None
python_model = None

if winner is None:
    winner_view = message_frame("No winner yet. Create a successful trial first, then rerun this section.")
else:
    winner_summary = trial.show_trial(winner.run_id, session=active)
    loaded_model = trial.load_model(winner.run_id, session=active)
    model_wrapper = loaded_model.unwrap_python_model()
    python_model = getattr(model_wrapper, "model", model_wrapper)
    winner_view = {
        "slug": winner.slug,
        "run_id": winner.run_id,
        "model_run_id": winner.run_id,
        "model_type": type(python_model).__name__,
        "preprocessor_type": type(python_model.preprocessor).__name__,
        "validation_status": winner_summary.tags.get("validation.status"),
        "deployment_ready": winner_summary.tags.get("deployment.ready"),
    }

winner_view


In [ ]:
training_report_view = (
    python_model.training_report()
    if python_model is not None
    else message_frame("No loaded model yet. Run the winner cell after at least one successful trial exists.")
)
training_report_view


In [ ]:
if python_model is None:
    registry = None
    feature_cols = []
    model_cols = []
    target_cols = []
else:
    registry = python_model.feature_registry
    feature_cols = registry.get_by_flag("feature")
    model_cols = registry.get_by_flag("model")
    target_cols = registry.get_by_flag("target")

pd.DataFrame(
    [
        {"group": "target", "count": len(target_cols), "sample": target_cols[:5]},
        {"group": "feature", "count": len(feature_cols), "sample": feature_cols[:5]},
        {"group": "model", "count": len(model_cols), "sample": model_cols[:5]},
    ]
)


In [ ]:
if registry is None:
    registry_view = message_frame("No feature registry yet because no model has been loaded.")
else:
    registry_df = registry.to_dataframe()
    registry_view = registry_df.loc[
        registry_df[["target", "feature", "model"]].any(axis=1),
        ["name", "dtype", "target", "feature", "model", "comments"],
    ].head(25)

registry_view


## Run Artifacts

Everything below is loaded from the selected run ID: manifest, evaluation report, feature registry, validation fixtures, and expected predictions.

In [ ]:
client = MlflowClient()


def list_artifact_paths(run_id: str, path: str = "") -> list[str]:
    paths = []
    for item in client.list_artifacts(run_id, path):
        if item.is_dir:
            paths.extend(list_artifact_paths(run_id, item.path))
        else:
            paths.append(item.path)
    return paths


artifact_paths = list_artifact_paths(winner.run_id) if winner is not None else []
artifact_view = pd.DataFrame({"artifact_path": artifact_paths})
artifact_view if not artifact_view.empty else message_frame("No run artifacts yet because no winner is selected.")


In [ ]:
def download_artifact(run_id: str, artifact_path: str) -> Path:
    return Path(client.download_artifacts(run_id, artifact_path))


def load_json_artifact(run_id: str, artifact_path: str) -> dict:
    return json.loads(download_artifact(run_id, artifact_path).read_text())


manifest = {}
eval_manifest = {}
eval_report = {}
validation_report = {}
model_report = {}
model_registry_artifact = pd.DataFrame()
dataset_registry_artifact = pd.DataFrame()

artifact_path_set = set(artifact_paths)

if winner is None:
    artifact_summary = message_frame("No artifact summary yet because no winner is selected.")
else:
    manifest = load_json_artifact(winner.run_id, "manifest.json") if "manifest.json" in artifact_path_set else {}
    eval_manifest = load_json_artifact(winner.run_id, "eval/manifest.json") if "eval/manifest.json" in artifact_path_set else {}
    primary_label = eval_manifest.get("primary_label")
    primary_report_path = f"eval/{primary_label}/report.json" if primary_label else ""
    eval_report = load_json_artifact(winner.run_id, primary_report_path) if primary_report_path in artifact_path_set else {}
    validation_report = (
        load_json_artifact(winner.run_id, "validation/report.json")
        if "validation/report.json" in artifact_path_set
        else {}
    )
    model_report = load_json_artifact(winner.run_id, "model/report.json") if "model/report.json" in artifact_path_set else {}
    if "features/feature_registry.csv" in artifact_path_set:
        model_registry_artifact = pd.read_csv(download_artifact(winner.run_id, "features/feature_registry.csv"))
    if "features/dataset_feature_registry.csv" in artifact_path_set:
        dataset_registry_artifact = pd.read_csv(download_artifact(winner.run_id, "features/dataset_feature_registry.csv"))
    artifact_summary = {
        "manifest_status": manifest.get("run", {}).get("trial_status"),
        "primary_metric": eval_report.get("primary"),
        "validation_status": validation_report.get("status"),
        "validation_rows": validation_report.get("row_count"),
        "validation_max_abs_diff": validation_report.get("max_abs_diff"),
        "model_report_available": bool(model_report),
        "model_registry_shape": model_registry_artifact.shape,
        "dataset_registry_shape": dataset_registry_artifact.shape,
    }

artifact_summary


## Load Predictions For An Eval Dataset

Use run-level eval entries as the pointer map, then load the exact eval dataset, augmentation manifests, and prediction artifacts recorded for that `eval_dataset_id`. Prediction artifacts stay under each run's `eval/<label>/` tree; the notebook follows the stored URIs instead of reconstructing GCS routes.

In [ ]:
eval_dataset = None
eval_dataset_entry = None
dataset_rows_with_augmentations = pd.DataFrame()
predictions_long = pd.DataFrame()
prediction_bundle = pd.DataFrame()

eval_entries = list(winner_summary.evaluations or ()) if winner_summary is not None else []
if not eval_entries:
    eval_dataset_view = message_frame("No eval entries yet. Run 4_reevaluate_existing_model.ipynb first.")
else:
    eval_dataset_entry = next(
        (entry for label in ("notebook_eval", "test") for entry in eval_entries if entry.label == label),
        eval_entries[0],
    )
    eval_dataset_id = eval_dataset_entry.eval_dataset_id
    eval_dataset = eval.load_eval_dataset(eval_dataset_id, session=active)
    eval_dataset_view = {
        "label": eval_dataset_entry.label,
        "eval_dataset_id": eval_dataset_id,
        "kind": eval_dataset.dataset.kind,
        "unique_key": eval_dataset.unique_key,
        "rows": len(eval_dataset.df),
        "record_uri": eval_dataset.dataset.record_gcs_uri,
        "current_run_predictions_uri": eval_dataset_entry.predictions_uri,
        "current_run_predictions_manifest_uri": eval_dataset_entry.predictions_manifest_uri,
    }

eval_dataset_view


In [ ]:
if eval_dataset is None:
    prediction_bundle_view = message_frame("No eval dataset selected yet.")
else:
    unique_key = list(eval_dataset.unique_key)
    prediction_frames = []
    prediction_index = []
    with automl_mlflow_client.bound_for(active, experiment_id=active.active_experiment_id):
        for row in leaderboard.rows:
            details = winner_summary if winner is not None and row.run_id == winner.run_id else trial.show_trial(row.run_id, session=active)
            for eval_entry in details.evaluations or ():
                if eval_entry.eval_dataset_id != eval_dataset.dataset.id:
                    continue
                try:
                    run = client.get_run(row.run_id)
                    model_run_name = run.data.tags.get("mlflow.runName") or row.slug or row.run_id
                except Exception:
                    model_run_name = row.slug or row.run_id
                predictions = mlflow_trial.artifacts.load_predictions(row.run_id, eval_entry.label)
                frame = predictions.frame.copy()
                frame.insert(0, "eval_label", eval_entry.label)
                frame.insert(0, "model_run_name", model_run_name)
                frame.insert(0, "model_run_id", row.run_id)
                prediction_frames.append(frame)
                prediction_index.append(
                    {
                        "model_run_id": row.run_id,
                        "model_run_name": model_run_name,
                        "label": eval_entry.label,
                        "predictions_uri": eval_entry.predictions_uri,
                        "predictions_manifest_uri": eval_entry.predictions_manifest_uri,
                        "rows": len(frame),
                    }
                )
    predictions_long = (
        pd.concat(prediction_frames, ignore_index=True)
        if prediction_frames
        else pd.DataFrame(columns=["model_run_id", "model_run_name", "eval_label", *unique_key, "y_pred"])
    )

    dataset_rows_with_augmentations = eval_dataset.df.copy()
    augmentation_index = [
        item
        for result in eval_entries
        if result.eval_dataset_id == eval_dataset.dataset.id
        for item in result.augmentations_used
    ]
    for item in augmentation_index:
        aug_frame = gcs.read_parquet(item["data_uri"])
        dataset_rows_with_augmentations = dataset_rows_with_augmentations.merge(
            aug_frame,
            on=unique_key,
            how="left",
        )

    prediction_bundle = dataset_rows_with_augmentations.merge(
        predictions_long,
        on=unique_key,
        how="left",
    )
    prediction_bundle_view = {
        "eval_dataset_id": eval_dataset.dataset.id,
        "dataset_rows": len(eval_dataset.df),
        "augmentation_count": len(augmentation_index),
        "prediction_file_count": len(prediction_index),
        "prediction_model_runs": [row["model_run_name"] for row in prediction_index],
        "predictions": prediction_index,
        "joined_shape": prediction_bundle.shape,
        "augmentations": augmentation_index,
    }

prediction_bundle_view


In [ ]:
prediction_bundle.head(20) if not prediction_bundle.empty else message_frame(
    "No prediction bundle rows yet."
)


## Validate Predictions From Artifacts

Load the run's validation input and expected output artifacts, score with the loaded model, and compare actual vs expected predictions.

In [ ]:
if winner is None or loaded_model is None:
    validation_view = message_frame("No validation artifacts yet because no winner is selected.")
else:
    validation_input = pd.read_parquet(download_artifact(winner.run_id, "validation/data/input.parquet"))
    validation_expected = pd.read_parquet(download_artifact(winner.run_id, "validation/data/expected.parquet"))
    validation_actual = np.asarray(loaded_model.predict(validation_input), dtype=float).reshape(-1)

    validation_check = validation_expected.assign(
        actual_score=validation_actual,
        abs_diff=lambda df: (df["actual_score"] - df["expected_score"]).abs(),
    )

    display(validation_check)
    validation_view = {
        "max_abs_diff": float(validation_check["abs_diff"].max()),
        "tolerance": validation_report.get("tolerance"),
        "passed": bool(validation_check["abs_diff"].max() <= validation_report.get("tolerance", 0.0)),
    }

validation_view


## Load This Run's Training Data

Load the exact train/test dataset slices recorded for the selected winner. This does not ask the project data module for the current active dataset; it reads the trial's logged dataset contract and verifies the loaded GCS data hashes by default.

In [ ]:
training_dataset = None
target_col = None
train_features = pd.DataFrame()
test_features = pd.DataFrame()

if winner is None:
    training_dataset_view = message_frame("No training dataset yet because no winner is selected.")
else:
    run_config = active.config.require_run_config()
    trial_id = f"{winner.trial_number}_{winner.slug}" if winner.trial_number is not None and winner.slug else winner.slug
    training_dataset = data.load_dataset_by_trial(trial_id, strict=True, session=active)
    target_col = training_dataset.registry.get_by_flag("target")[0]
    train = data.load_dataset_by_trial(trial_id, split_name=run_config.train_split, strict=True, session=active)
    holdout = data.load_dataset_by_trial(trial_id, split_name=run_config.eval_split, strict=True, session=active)
    train_features = train.df.drop(columns=[target_col])
    test_features = holdout.df.drop(columns=[target_col])
    training_dataset_view = {
        "source": "trial_contract",
        "trial_id": trial_id,
        "run_id": winner.run_id,
        "dataset_id": training_dataset.dataset.id,
        "identity_hash": training_dataset.dataset.identity_hash,
        "train_shape": train.df.shape,
        "test_shape": holdout.df.shape,
        "target_col": target_col,
        "train_feature_shape": train_features.shape,
        "test_feature_shape": test_features.shape,
    }

training_dataset_view


In [ ]:
if loaded_model is None or training_dataset is None or target_col is None:
    prediction_view = message_frame("No sample predictions yet because no winner is selected.")
else:
    sample_predictions = loaded_model.predict(test_features.head(10))
    prediction_view = pd.DataFrame(
        {
            "row_index": test_features.head(10).index,
            "target": holdout.df[target_col].head(10).to_numpy(),
            "prediction": np.asarray(sample_predictions, dtype=float),
        }
    )

prediction_view


## Code Path Caution

MLflow loads the model with the logged model code that was logged with that run. That can add archived `model/code` paths to `sys.path`. Keep project imports near the top of the notebook; use subprocess isolation if you need to score multiple historical logged model code versions with conflicting code.

In [ ]:
model_code_paths = [
    path
    for path in sys.path
    if "/artifacts/" in path and path.endswith("/model/code")
]

{
    "project": active.project_name,
    "project_dir": str(config.project_dir),
    "model_code_paths_added_by_mlflow": model_code_paths[:5],
}
